Importing all necessary libraries

In [1]:
#Standard Libraries
import datetime
import re
import os

#Third-Party Libraries
import numpy as np
import pandas as pd
import requests
import torch
import torch.nn.functional as F
from tqdm import tqdm

#Transformers/NLP
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    logging
)


C:\FYP\newEnv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Code to fetch news articles from the Alpha Vantage API from the last stored publication till now.

In [2]:
API_KEY = "TED7KR7H5HP2E5YG" 
CSV_FILE = "C:\\FYP\\sxk1332\\Next Day Predictions\\apple_news_10yrs.csv"

#Determine the exact gap of missing days in the current csv file
df = pd.read_csv(CSV_FILE)
df['time_published_dt'] = pd.to_datetime(df['time_published'], format='%Y%m%dT%H%M%S')
    
last_date = df['time_published_dt'].max()
now = datetime.datetime.now()
   
#Format the data
time_from = (last_date + datetime.timedelta(minutes=1)).strftime("%Y%m%dT%H%M")
time_to = now.strftime("%Y%m%dT%H%M")

print(f"Fetching news from {last_date} to {now}")

#Make the request from the API
url = "https://www.alphavantage.co/query"
params = {
    "function": "NEWS_SENTIMENT",
    "tickers": "AAPL",
    "time_from": time_from,
    "time_to": time_to,
    "limit": 1000,
    "apikey": API_KEY,
    "sort": "LATEST"
}

response = requests.get(url, params=params).json()
    
#Adding the new finaical news to the exisiting csv file
if "feed" in response and response["feed"]:
    new_articles = response["feed"]
    new_df = pd.DataFrame(new_articles)

    df = (
        pd.concat([df.drop(columns=["time_published_dt"], errors="ignore"), new_df])
        .drop_duplicates(subset=['url'])
        .reset_index(drop=True)
    )

    df.to_csv(CSV_FILE, index=False)
    print(f"Successfully added {len(new_articles)} new articles!")
else:
    print("No new articles found.")

Fetching news from 2026-04-28 07:55:59 to 2026-04-28 21:12:10.704995


Successfully added 17 new articles!


Filtering the original CSV file to only include Apple related articles

In [3]:
#Define the filter pattern
apple_pattern = re.compile(
    r"\b(apple|iphone|ipad|mac|macbook|airpods|tim cook|steve jobs|app store|ios|macos)\b",
    re.IGNORECASE
)

#Filter the news titles for Apple-related words and remove duplicates
news_df = (
    df[df["title"].apply(lambda x: bool(apple_pattern.search(str(x))))].drop_duplicates(subset="title").assign
    (
        date=lambda x: pd.to_datetime(
            x["time_published"],
            format="%Y%m%dT%H%M%S"
        )
    )
    .sort_values("date")
    .reset_index(drop=True)
)

Loading the FinBert model (a pre-trained specifically for financial sentiment)

In [4]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"
logging.set_verbosity_error()

MODEL_NAME = "ProsusAI/finbert"

#Loading the tokenizer specifically designed for FinBERT
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

#Loading the pre-trained FinBERT model
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

#Switching the model to evaludation mode  
model.eval()

Loading weights:   0%|                                                                                                             | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 20091.40it/s]

BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

Running FinBERT on each news title and calculating their sentiment scores

In [5]:
def finbert_sentiment_batch(texts, batch_size=32):
    all_probs = []

    #Grouping articles makes processing faster
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        )

        #Passing inputs through the FinBERT model to convert predictions into probabilties
        with torch.no_grad():
            outputs = model(**inputs)
            probs = F.softmax(outputs.logits, dim=1)

        all_probs.extend(probs.tolist())

    return pd.DataFrame(all_probs, columns=["positive", "negative", "neutral"])

#Dropping existing sentiment columns to avoid duplicates
news_df = news_df.drop(columns=["positive", "negative", "neutral", "sentiment_score"], errors="ignore")

#Applying the finbert_sentiment function to every title
sentiment_df = finbert_sentiment_batch(news_df["title"].tolist())
news_df = pd.concat([news_df, sentiment_df], axis=1)

#Calculate the sentiment score per aticle
news_df["sentiment_score"] = news_df["positive"] - news_df["negative"]

print(news_df.head())

                                               title  \
0  Apple 'failing to protect Chinese factory work...   
1             Chevron to Bring Apple Pay to the Pump   
2  Following Paltalk acquisition, Tinychat goes f...   
3         What Unilever shares with Google and Apple   
4  Infosys CEO Vishal Sikka gifted 3000 sets of i...   

                                                 url   time_published  \
0         https://www.bbc.com/news/business-30532463  20141218T095612   
1  https://cspdailynews.com/technologyservices/ch...  20141230T102220   
2  https://appadvice.com/appnn/2014/12/following-...  20141230T105131   
3  https://fortune.com/2015/01/07/what-unilever-s...  20150107T094251   
4  https://www.indiatoday.in/education-today/gk-a...  20150110T144700   

               authors                                            summary  \
0   ['Richard Bilton']  An undercover BBC Panorama investigation found...   
1                   []  Chevron announced it is working with Apple to 

Daily Aggregation

In [6]:
#Calculating average sentiment scores for each article
daily_sentiment = (news_df.groupby(news_df["date"].dt.date)
    .agg(avg_sentiment=("sentiment_score", "mean"),)
    .reset_index()
)

#Filtering the dataframe to start from 2015-01-01 to match the model start dates
daily_sentiment = daily_sentiment[daily_sentiment["date"] >= pd.to_datetime("2015-01-01").date()]

#Save the final dataframe to a csv file
daily_sentiment.to_csv("C:\\FYP\\sxk1332\\Next Day Predictions\\daily_apple_news_sentiment.csv", index=False)

print("New articles added successfully")

New articles added successfully
